In [8]:
import os
import cv2
import numpy as np
from collections import Counter

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [9]:

DATASET_PATH = "DataSet_Face"

#  Label mapping
label_map = {
    "angry": 0,
    "disgust": 1,
    "fear": 2,
    "happy": 3,
    "sad": 4,
    "surprise": 5,
    "neutral": 6
}

IMG_SIZE = 96

X = []
y = []

# 🔄 Load and clean dataset
for label_name in os.listdir(DATASET_PATH):
    if label_name not in label_map:
        continue

    label = label_map[label_name]
    folder_path = os.path.join(DATASET_PATH, label_name)

    for file in os.listdir(folder_path):
        img_path = os.path.join(folder_path, file)

        img = cv2.imread(img_path)

        # ❌ skip bad images
        if img is None:
            continue

        try:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            img = img / 255.0

            X.append(img)
            y.append(label)
        except:
            continue


In [10]:



# 📏 Convert to numpy
X = np.array(X).reshape(-1, IMG_SIZE, IMG_SIZE, 1)
y = np.array(y)

print("Dataset shape:", X.shape)
print("Label distribution:", Counter(y))

# 🧠 Build CNN model with Softmax
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(7, activation='softmax')  # 🔥 Softmax layer
])

# ⚙️ Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 🧪 Train
model.fit(X, y, epochs=15, batch_size=32, validation_split=0.2)

# 💾 Save model
model.save("emotion_model.h5")

print("✅ Training complete. Model saved as emotion_model.h5")

Dataset shape: (78488, 96, 96, 1)
Label distribution: Counter({3: 18613, 6: 13131, 4: 11365, 2: 10017, 0: 9915, 5: 9091, 1: 6356})
Epoch 1/15
1963/1963 ━━━━━━━━━━━━━━━━━━━━ 67s 33ms/step - accuracy: 0.4026 - loss: 1.4780 - val_accuracy: 0.0055 - val_loss: 9.3149
Epoch 2/15
1963/1963 ━━━━━━━━━━━━━━━━━━━━ 68s 35ms/step - accuracy: 0.5796 - loss: 1.0859 - val_accuracy: 0.0133 - val_loss: 12.0770
Epoch 3/15
1963/1963 ━━━━━━━━━━━━━━━━━━━━ 69s 35ms/step - accuracy: 0.6157 - loss: 0.9925 - val_accuracy: 0.0170 - val_loss: 13.6282
Epoch 4/15
1963/1963 ━━━━━━━━━━━━━━━━━━━━ 82s 42ms/step - accuracy: 0.6391 - loss: 0.9301 - val_accuracy: 0.0431 - val_loss: 16.3442
Epoch 5/15
1963/1963 ━━━━━━━━━━━━━━━━━━━━ 100s 51ms/step - accuracy: 0.6580 - loss: 0.8808 - val_accuracy: 0.0173 - val_loss: 19.3144
Epoch 6/15
1963/1963 ━━━━━━━━━━━━━━━━━━━━ 90s 46ms/step - accuracy: 0.6777 - loss: 0.8265 - val_accuracy: 0.0259 - val_loss: 24.8365
Epoch 7/15
1963/1963 ━━━━━━━━━━━━━━━━━━━━ 90s 46ms/step - accuracy: 0.6

✅ Training complete. Model saved as emotion_model.h5
